First we load in all the data

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.special import expit

DATA_DIR = Path(".")

X_trn_df = pd.read_csv(DATA_DIR / "X_trn.csv")
y_trn_df = pd.read_csv(DATA_DIR / "y_trn.csv")
X_test_df = pd.read_csv(DATA_DIR / "X_test.csv")

X_trn = X_trn_df.to_numpy(dtype=np.float64)
Y_trn = y_trn_df.to_numpy(dtype=np.float64)
X_test = X_test_df.to_numpy(dtype=np.float64)

print(X_trn_df.shape)
print(y_trn_df.shape)
print(X_test_df.shape)

print(X_trn_df.head())
print(y_trn_df.head())

(25000, 23)
(25000, 1)
(5000, 23)
    X01    X02  X03  X04  X05  X06  X07   X08   X09    X10  ...   X14  X15  \
0  1514  30177  -10    1   26   -5   -4     0  1890  33490  ...  3835   -2   
1     0      0  -16    2   51  -29  -16     0     0      0  ...     0  -12   
2     0   2945  -16    1   39  -23  -13  1216  2997   2675  ...  2470  -10   
3  1064  51869  -19    1   36  -17  -10  1900  2176  30815  ...  1039   -8   
4  1506  54456  -19    2   23  -17  -10  2009  1744  53609  ...  1776   -8   

   X16  X17    X18     X19   X20   X21  X22    X23  
0    2   -3  28439   51533  3097  1896   22  35014  
1    1  -23      0  296630     0     0   -2      0  
2    1  -19   1271  306063     0     0    2     -4  
3    1  -15  47381   53052  1916  1134    6  30258  
4    1  -15  50334   47970  1820  1624    6  45063  

[5 rows x 23 columns]
   Y
0  1
1  0
2  0
3  1
4  0


In [10]:
import numpy as np

rng = np.random.default_rng(seed=42)

n_features = X_trn.shape[1]
n_hidden = 1

W1 = rng.normal(
    loc=0.0,
    scale=np.sqrt(2 / n_features),
    size=(n_features, n_hidden)
)

b1 = np.zeros((1, n_hidden))

W2 = rng.normal(
    loc=0.0,
    scale=np.sqrt(2 / n_hidden),
    size=(n_hidden, 1)
)

b2 = np.zeros((1, 1))

print(W1.shape)
print(b1.shape)
print(W2.shape)
print(b2.shape)

(23, 1)
(1, 1)
(1, 1)
(1, 1)


In [ ]:
# Forward pass
z1 = W1.T @ X_trn.T + b1.T
h = np.maximum(0, z1)

z2 = W2.T @ h + b2
p = expit(z2).T

print(p.shape)

BS = 1 / len(p) * np.sum((Y_trn - p) ** 2)
print(BS)

# Derivatives
relu_derivative = (z1 > 0).astype(float).T

print(relu_derivative.shape)

print("Y trn", Y_trn.shape)
print("p", p.shape)
print("W2", W2.shape)
print("h", h.shape)
print("relu_derivative", relu_derivative.shape)
print("X_trn", X_trn.shape)

n = len(p)

# dL/dz2, shape: (n, 1)
dz2 = -2 * (Y_trn - p) * p * (1 - p) / n

# Gradients for output layer
gradient_W2 = h @ dz2
gradient_b2 = np.sum(dz2, axis=0, keepdims=True)

# dL/dz1, shape: (n, 1)
dz1 = dz2 * W2 * relu_derivative

# Gradients for hidden layer
gradient_W1 = X_trn.T @ dz1
gradient_b1 = np.sum(dz1, axis=0, keepdims=True)

gradient_matrix = np.concatenate(
    (
        gradient_W1.flatten(),
        gradient_b1.flatten(),
        gradient_W2.flatten(),
        gradient_b2.flatten()
    )
)

print("dz1", dz1.shape)
print("dz2", dz2.shape)
print("gradient_W1", gradient_W1.shape)
print("gradient_b1", gradient_b1.shape)
print("gradient_W2", gradient_W2.shape)
print("gradient_b2", gradient_b2.shape)

eta = 0.01

W1 -= eta * gradient_W1
b1 -= eta * gradient_b1
W2 -= eta * gradient_W2
b2 -= eta * gradient_b2

(25000, 1)
0.22165593246405854
(25000, 1)
Y trn (25000, 1)
p (25000, 1)
W2 (1, 1)
h (1, 25000)
relu_derivative (25000, 1)
X_trn (25000, 23)
dz1 (25000, 1)
dz2 (25000, 1)
gradient_W1 (23, 1)
gradient_b1 (1, 1)
gradient_W2 (1, 1)
gradient_b2 (1, 1)
gradient_matrix (26,)


In [176]:
def train_neural_network(X_trn, Y_trn, W1, b1, W2, b2, eta=0.01, iterations=1000):
    Loss = []

    for i in range(iterations):
        z1 = W1.T @ X_trn.T + b1.T
        h = np.maximum(0, z1)

        z2 = W2.T @ h + b2
        p = expit(z2).T

        # Derivatives
        relu_derivative = (z1 > 0).astype(float).T

        n = len(p)

        # dL/dz2, shape: (n, 1)
        dz2 = -2 * (Y_trn - p) * p * (1 - p) / n

        # Gradients for output layer
        gradient_W2 = h @ dz2
        gradient_b2 = np.sum(dz2, axis=0, keepdims=True)

        # dL/dz1, shape: (n, 1)
        dz1 = dz2 * W2.T * relu_derivative

        # Gradients for hidden layer
        gradient_W1 = X_trn.T @ dz1
        gradient_b1 = np.sum(dz1, axis=0, keepdims=True)

        gradient_matrix = np.concatenate(
            (
                gradient_W1.flatten(),
                gradient_b1.flatten(),
                gradient_W2.flatten(),
                gradient_b2.flatten()
            )
        )

        W1 -= eta * gradient_W1
        b1 -= eta * gradient_b1
        W2 -= eta * gradient_W2
        b2 -= eta * gradient_b2

        BS = 1 / len(p) * np.sum((Y_trn - p) ** 2)
        Loss.append(BS)

        #if i % 100 == 0:
        #    print(f"Iteration {i}: Loss = {BS}")
        #    print("z1:", z1)
        #    print("z2:", z2)
        #    print("p:", p)

    print("Begin Loss:", Loss[0])
    print("Final Loss:", Loss[-1])
    return W1, b1, W2, b2, Loss

In [178]:
rng = np.random.default_rng()

n_features = X_trn.shape[1]
n_hidden = 5

W1 = rng.normal(
    loc=0.0,
    scale=np.sqrt(2 / n_features),
    size=(n_features, n_hidden)
)

b1 = np.zeros((1, n_hidden))

W2 = rng.normal(
    loc=0.0,
    scale=np.sqrt(2 / n_hidden),
    size=(n_hidden, 1)
)

b2 = np.zeros((1, 1))

base_rate = np.mean(Y_trn)
baseline_brier = np.mean((Y_trn - base_rate) ** 2)

print("base_rate:", base_rate)
print("baseline_brier:", baseline_brier)

X_trn_mean = np.mean(X_trn, axis=0)
X_trn_std = np.std(X_trn, axis=0)
X_trn_scaled = (X_trn - X_trn_mean) / X_trn_std

X_test_mean = np.mean(X_test, axis=0)
X_test_std = np.std(X_test, axis=0)
X_test_scaled = (X_test - X_test_mean) / X_test_std


W1, b1, W2, b2, Loss = train_neural_network(X_trn_scaled, Y_trn, W1, b1, W2, b2)
check_on_test_set(X_test_scaled, Y_trn, W1, b1, W2, b2)

base_rate: 0.2212
baseline_brier: 0.17227056000000004
Begin Loss: 0.23125013349184848
Final Loss: 0.17909374697645333


np.float64(0.18201515266181087)

In [137]:
def check_on_test_set(X_test, Y_trn, W1, b1, W2, b2):
    z1 = W1.T @ X_test.T + b1.T
    h = np.maximum(0, z1)

    z2 = W2.T @ h + b2
    p = expit(z2).T

    BS = 1 / len(p) * np.sum((Y_trn[:len(p)] - p) ** 2)

    return BS

In [190]:
import numpy as np
from scipy.special import expit

def train_val_split(X, Y, val_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    indices = rng.permutation(n)

    n_val = int(val_fraction * n)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    return X[train_idx], Y[train_idx], X[val_idx], Y[val_idx]


def fit_standardizer(X):
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True)
    sigma[sigma == 0] = 1.0
    return mu, sigma


def apply_standardizer(X, mu, sigma):
    return (X - mu) / sigma


def initialize_parameters(n_features, n_hidden, seed=42):
    rng = np.random.default_rng(seed)

    W1 = rng.normal(0.0, np.sqrt(2 / n_features), size=(n_features, n_hidden))
    b1 = np.zeros((1, n_hidden))

    W2 = rng.normal(0.0, np.sqrt(2 / n_hidden), size=(n_hidden, 1))
    b2 = np.zeros((1, 1))

    return W1, b1, W2, b2


def predict_neural_network(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    h = np.maximum(0, z1)

    z2 = h @ W2 + b2
    p = expit(z2)

    return p


def brier_score(Y, p):
    return np.mean((Y - p) ** 2)


def train_neural_network_minibatch(
    X,
    Y,
    W1,
    b1,
    W2,
    b2,
    eta=0.01,
    iterations=1000,
    batch_size=256,
    seed=42
):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    Loss = []

    for i in range(iterations):
        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]

            X_batch = X[batch_idx]
            Y_batch = Y[batch_idx]
            m = X_batch.shape[0]

            # Forward pass
            z1 = X_batch @ W1 + b1
            h = np.maximum(0, z1)

            z2 = h @ W2 + b2
            p = expit(z2)

            # Backward pass
            dz2 = 2 * (p - Y_batch) * p * (1 - p) / m

            gradient_W2 = h.T @ dz2
            gradient_b2 = np.sum(dz2, axis=0, keepdims=True)

            relu_derivative = (z1 > 0).astype(float)
            dz1 = (dz2 @ W2.T) * relu_derivative

            gradient_W1 = X_batch.T @ dz1
            gradient_b1 = np.sum(dz1, axis=0, keepdims=True)

            # Update
            W1 -= eta * gradient_W1
            b1 -= eta * gradient_b1
            W2 -= eta * gradient_W2
            b2 -= eta * gradient_b2

        # Track full training loss after each epoch
        p_train = predict_neural_network(X, W1, b1, W2, b2)
        Loss.append(brier_score(Y, p_train))

    return W1, b1, W2, b2, Loss


# 1. Maak train/validation split uit de originele trainingdata
X_fit_raw, Y_fit, X_val_raw, Y_val = train_val_split(
    X_trn,
    Y_trn,
    val_fraction=0.2,
    seed=42
)

# 2. Fit scaling alleen op fit/train-split
mu_val, sigma_val = fit_standardizer(X_fit_raw)

X_fit = apply_standardizer(X_fit_raw, mu_val, sigma_val)
X_val = apply_standardizer(X_val_raw, mu_val, sigma_val)

# 3. Test verschillende learning rates
eta_grid = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 0.0003]

n_features = X_fit.shape[1]
n_hidden = 4
batch_size = 256
iterations = 1000

results = []

for eta in eta_grid:
    W1, b1, W2, b2 = initialize_parameters(
        n_features=n_features,
        n_hidden=n_hidden,
        seed=42
    )

    W1, b1, W2, b2, Loss = train_neural_network_minibatch(
        X_fit,
        Y_fit,
        W1,
        b1,
        W2,
        b2,
        eta=eta,
        iterations=iterations,
        batch_size=batch_size,
        seed=42
    )

    p_val = predict_neural_network(X_val, W1, b1, W2, b2)
    val_brier = brier_score(Y_val, p_val)

    results.append((eta, val_brier, Loss[-1]))

    print("eta:", eta, "| validation Brier:", val_brier, "| train Brier:", Loss[-1])

# 4. Kies beste eta op basis van validation Brier
best_eta, best_val_brier, best_train_brier = min(results, key=lambda x: x[1])

print("\nBeste eta:", best_eta)
print("Beste validation Brier:", best_val_brier)


# 5. Final model trainen op alle trainingdata met beste eta
mu_final, sigma_final = fit_standardizer(X_trn)

X_trn_scaled = apply_standardizer(X_trn, mu_final, sigma_final)
X_test_scaled = apply_standardizer(X_test, mu_final, sigma_final)

W1, b1, W2, b2 = initialize_parameters(
    n_features=X_trn_scaled.shape[1],
    n_hidden=n_hidden,
    seed=42
)

W1, b1, W2, b2, Loss_final = train_neural_network_minibatch(
    X_trn_scaled,
    Y_trn,
    W1,
    b1,
    W2,
    b2,
    eta=best_eta,
    iterations=iterations,
    batch_size=batch_size,
    seed=42
)

p_test = predict_neural_network(X_test_scaled, W1, b1, W2, b2)

print("\nFinal train Brier:", Loss_final[-1])
print("p_test shape:", p_test.shape)
print("p_test min/max/mean:", p_test.min(), p_test.max(), p_test.mean())
print(p_test)

eta: 1.0 | validation Brier: 0.15062343313183116 | train Brier: 0.12422010120403522
eta: 0.3 | validation Brier: 0.1475214328793972 | train Brier: 0.12411316938608268
eta: 0.1 | validation Brier: 0.14007084376878323 | train Brier: 0.12846419030732184
eta: 0.03 | validation Brier: 0.13705320877550428 | train Brier: 0.13334052149317002
eta: 0.01 | validation Brier: 0.13777375240468764 | train Brier: 0.13569041791643946
eta: 0.003 | validation Brier: 0.14135188401475077 | train Brier: 0.1402996202068894
eta: 0.001 | validation Brier: 0.1472324597022815 | train Brier: 0.1462892118380016
eta: 0.0003 | validation Brier: 0.15520198558877302 | train Brier: 0.15415962406169453

Beste eta: 0.03
Beste validation Brier: 0.13705320877550428

Final train Brier: 0.1334758022217965
p_test shape: (5000, 1)
p_test min/max/mean: 9.980733057313686e-32 0.999914196801113 0.22613977900942836
[[0.56487797]
 [0.27004048]
 [0.73404023]
 ...
 [0.16970644]
 [0.29955138]
 [0.14992628]]


In [184]:
p_val = predict_neural_network(X_val, W1, b1, W2, b2)

print("val p min/max/mean:", p_val.min(), p_val.max(), p_val.mean())
print("val y mean:", Y_val.mean())
print("p < 0.01:", np.mean(p_val < 0.01))
print("p > 0.99:", np.mean(p_val > 0.99))

val p min/max/mean: 4.7667651193704505e-14 0.999731674823586 0.23328708619937405
val y mean: 0.219
p < 0.01: 0.0196
p > 0.99: 0.0016
